# Saving and Loading a Trained Pipeline

After training a pipeline you typically want to **persist** it so it can be
reloaded later — on another machine, in a different process, or simply to
avoid retraining.

A TSUT pipeline is fully captured by two artefacts:

| Artefact | What it contains | Format |
|----------|-----------------|--------|
| **Config** | Node types, ports, edges, hyperparameters | JSON (via Pydantic `model_dump_json`) |
| **Trained parameters** | Fitted weights, scaler stats, etc. | Pickle (via `save_params_to_dir`) |

This notebook walks through the full cycle:

1. Build and train a pipeline with `InputsPassthrough`
2. Save the config (JSON) and trained parameters (pickle) to disk
3. Reload them in a fresh pipeline and verify inference produces the same results

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure

configure(level="INFO", stream=sys.stdout, fmt="text")

SAVE_DIR = Path("/tmp/tsut_saved_pipeline")
SAVE_DIR.mkdir(exist_ok=True)

## 2. Synthetic dataset

In [ ]:
rng = np.random.default_rng(42)
n = 200

X_df = pd.DataFrame({"x1": rng.standard_normal(n), "x2": rng.standard_normal(n)})
y_df = pd.DataFrame({"target": 3.0 * X_df["x1"] + 2.0 * X_df["x2"] + rng.normal(0, 0.05, n)})

print(f"X: {X_df.shape}, y: {y_df.shape}")

## 3. Build and train the pipeline

In [ ]:
from tsut.core.common.data.data import (
    DATA_CATEGORY_MAPPING,
    ArrayLikeEnum,
    DataCategoryEnum,
    DataStructureEnum,
    TabularDataContext,
)
from tsut.core.common.enums import NodeExecutionMode
from tsut.core.nodes.node import Port
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig
from tsut.core.pipeline.runners.smart_runner import SmartRunner

source_config = NODE_REGISTRY.get_node_config_class("InputsPassthrough")(
    out_ports={
        "X": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.NUMERICAL,
            data_shape="batch feature",
            desc="Feature matrix.",
        ),
        "y": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.NUMERICAL,
            data_shape="batch 1",
            desc="Targets (training/evaluation only).",
            mode=[NodeExecutionMode.TRAINING, NodeExecutionMode.EVALUATION],
        ),
    },
)

nodes = {
    "source": ("InputsPassthrough", source_config),
    "scaler": ("StandardScaler", NODE_REGISTRY.get_node_config_class("StandardScaler")()),
    "model": ("LinearRegression", NODE_REGISTRY.get_node_config_class("LinearRegression")()),
    "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
    "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

edges = [
    Edge(source="source", target="scaler", ports_map=[("X", "input")]),
    Edge(source="scaler", target="model", ports_map=[("output", "X")]),
    Edge(source="source", target="model", ports_map=[("y", "y")]),
    Edge(source="model", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="source", target="r2", ports_map=[("y", "target")]),
    Edge(source="model", target="sink", ports_map=[("pred", "dump")]),
]

pipe = Pipeline(config=PipelineConfig(nodes=nodes, edges=edges, name="SaveLoadDemo"))
pipe.compile()
runner = SmartRunner(pipe)

In [ ]:
NUM = DATA_CATEGORY_MAPPING[DataCategoryEnum.NUMERICAL]

X_ctx = TabularDataContext(
    columns=list(X_df.columns),
    dtypes=[X_df[c].dtype for c in X_df.columns],
    categories=[NUM] * len(X_df.columns),
)
y_ctx = TabularDataContext(
    columns=list(y_df.columns),
    dtypes=[y_df[c].dtype for c in y_df.columns],
    categories=[NUM],
)

X_pair = (X_df, X_ctx)
y_pair = (y_df, y_ctx)

runner.train(input_data={"source": {"X": X_pair, "y": y_pair}})
print("Training complete.")

Quick sanity check — run inference and note the predictions so we can
compare after reloading.

In [ ]:
original_preds = runner.infer(input_data={"source": {"X": X_pair}})
original_pred_df = original_preds["dump"][0]
print(f"Original predictions shape: {original_pred_df.shape}")
original_pred_df.head()

## 4. Save to disk

The `Pipeline` class provides three save methods that all use the same
file-safe naming convention: `{sanitized_name}_v{version}_<suffix>`.

| Method | File produced | Contents |
|--------|--------------|----------|
| `save_config_to_dir()` | `…_config.json` | Full pipeline topology and node configs |
| `save_params_to_dir()` | `…_params.pkl` | Trained weights for every parameterised node |
| `save_html_to_dir()` | `…_graph.html` | Interactive Plotly graph of the DAG |

In [ ]:
pipe.save_config_to_dir(str(SAVE_DIR))
pipe.save_params_to_dir(str(SAVE_DIR))
pipe.save_html_to_dir(str(SAVE_DIR))

print(f"File basename: {pipe.file_basename}")
for path in sorted(SAVE_DIR.iterdir()):
    print(f"  {path.name}  ({path.stat().st_size} bytes)")

Let's peek at the saved config to see what's inside.

In [ ]:
import json

config_path = SAVE_DIR / f"{pipe.file_basename}_config.json"
saved_config = json.loads(config_path.read_text())
print("Pipeline name:", saved_config["name"])
print("Nodes:", list(saved_config["nodes"].keys()))
print("Number of edges:", len(saved_config["edges"]))

We can also inspect which nodes have trained parameters.

In [ ]:
params = pipe.get_params()
for node_name, node_params in params.items():
    param_keys = list(node_params.keys()) if isinstance(node_params, dict) else type(node_params).__name__
    print(f"  {node_name}: {param_keys}")

## 5. Load from disk and recreate the pipeline

To reconstruct a trained pipeline from saved artefacts:

1. **Parse** the JSON config into a dict
2. **Reconstruct** node configs using the registry (each node type has its own concrete config class — we need the registry to deserialize into the right subclass)
3. **Create** a new `Pipeline` from the reconstructed config
4. **Compile** it (this instantiates the node objects)
5. **Load** the trained parameters into the node objects

In [ ]:
raw = json.loads(config_path.read_text())

# Pydantic deserialises every node config as the base NodeConfig,
# losing subclass-specific fields (running_config, hyperparameters, …).
# We reconstruct each config through the registry's concrete class.
reconstructed_nodes: dict[str, tuple[str, None]] = {}
for node_name, (node_type, _config_dict) in raw["nodes"].items():
    config_cls = NODE_REGISTRY.get_node_config_class(node_type)
    if _config_dict is not None:
        node_config = config_cls.model_validate(_config_dict)
    else:
        node_config = config_cls()
    reconstructed_nodes[node_name] = (node_type, node_config)

loaded_config = PipelineConfig(
    nodes=reconstructed_nodes,
    edges=[Edge(**e) for e in raw["edges"]],
    name=raw["name"],
    version=raw.get("version", {}),
)

loaded_pipe = Pipeline(config=loaded_config)
loaded_pipe.compile()
loaded_pipe.load_params_from_dir(str(SAVE_DIR))

print(f"Loaded pipeline: {loaded_pipe.name} v{loaded_pipe.version}")
print(f"Nodes: {list(loaded_pipe.node_objects.keys())}")

## 6. Verify: inference matches the original

The reloaded pipeline should produce **identical** predictions to the
original, since the config and trained weights are the same.

In [ ]:
loaded_runner = SmartRunner(loaded_pipe)

loaded_preds = loaded_runner.infer(input_data={"source": {"X": X_pair}})
loaded_pred_df = loaded_preds["dump"][0]

print(f"Loaded predictions shape: {loaded_pred_df.shape}")
loaded_pred_df.head()

In [ ]:
max_diff = np.max(np.abs(original_pred_df.values - loaded_pred_df.values))
print(f"Max absolute difference between original and loaded predictions: {max_diff}")

np.testing.assert_allclose(original_pred_df.values, loaded_pred_df.values, atol=1e-12)
print("Predictions are identical.")

The loaded pipeline can also evaluate metrics, just like the original.

In [ ]:
metrics = loaded_runner.evaluate(input_data={"source": {"X": X_pair, "y": y_pair}})

for name, (score_df, _) in metrics.items():
    print(f"{name}: {float(score_df.to_numpy().flatten()[0]):.6f}")

## 7. Cleanup

In [ ]:
import shutil

shutil.rmtree(SAVE_DIR)
print(f"Cleaned up {SAVE_DIR}")

## Summary

```python
# --- Save (all files share the same sanitized basename) ---
pipeline.save_config_to_dir("./saved/")   # {basename}_config.json
pipeline.save_params_to_dir("./saved/")   # {basename}_params.pkl
pipeline.save_html_to_dir("./saved/")     # {basename}_graph.html

# --- Load ---
raw = json.loads(Path("./saved/{basename}_config.json").read_text())

# Reconstruct node configs through the registry so concrete
# subclass fields (running_config, hyperparameters, ports) survive.
nodes = {}
for name, (node_type, cfg_dict) in raw["nodes"].items():
    config_cls = NODE_REGISTRY.get_node_config_class(node_type)
    nodes[name] = (node_type, config_cls.model_validate(cfg_dict) if cfg_dict else config_cls())

config = PipelineConfig(
    nodes=nodes,
    edges=[Edge(**e) for e in raw["edges"]],
    name=raw["name"],
    version=raw.get("version", {}),
)

pipeline = Pipeline(config=config)
pipeline.compile()
pipeline.load_params_from_dir("./saved/")
```

The config JSON is human-readable and diffable; the params pickle holds
arbitrary fitted state (numpy arrays, sklearn internals, torch state dicts).
The registry lookup at load time ensures each node config deserialises into
its correct concrete class.